# 04 · Preparación y exportación de las tablas

Conservamos una tabla por asignatura. Documentamos las decisiones de limpieza, revisamos valores extremos y exportamos los CSV que se utilizan en el informe.

## 1. Librerías y carga de datos

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

raiz = Path.cwd()
if not (raiz / 'datos').is_dir():
    raiz = raiz.parent
if not (raiz / 'datos').is_dir():
    raise FileNotFoundError('Abre el proyecto o la carpeta notebooks antes de ejecutar.')
resultados = raiz / 'resultados'
resultados.mkdir(exist_ok=True)
fuentes = {'matematicas': 'student-mat.csv', 'portugues': 'student-por.csv'}
datos = {nombre: pd.read_csv(raiz / 'datos' / archivo, sep=';')
         for nombre, archivo in fuentes.items()}


## 2. Valores faltantes y duplicados
No imputamos datos ni eliminamos filas si la revisión no muestra esos problemas.

In [2]:
calidad = pd.DataFrame([
    {'Asignatura': nombre.capitalize(), 'Registros': len(df),
     'Nulos': int(df.isna().sum().sum()), 'Duplicados exactos': int(df.duplicated().sum())}
    for nombre, df in datos.items()
])
display(calidad)
assert calidad['Nulos'].sum() == 0 and calidad['Duplicados exactos'].sum() == 0


,Asignatura,Registros,Nulos,Duplicados exactos
0,Matematicas,395,0,0
1,Portugues,649,0,0


## 3. Ausencias alejadas del rango central
Usamos la regla Q3 + 1,5 × RIC como señal de revisión. Un valor señalado no es necesariamente un error.

In [3]:
diagnostico = []
for nombre, df in datos.items():
    q1, q3 = df['absences'].quantile([0.25, 0.75])
    limite = q3 + 1.5 * (q3 - q1)
    diagnostico.append({'Asignatura': nombre.capitalize(), 'Q1': q1, 'Q3': q3,
                        'Límite superior': limite, 'Registros por encima': int((df['absences'] > limite).sum())})
display(pd.DataFrame(diagnostico))


,Asignatura,Q1,Q3,Límite superior,Registros por encima
0,Matematicas,0.0,8.0,20.0,15
1,Portugues,0.0,6.0,15.0,21


Conservamos todos los registros porque no tenemos evidencia de que las ausencias altas o las notas cero sean errores. La revisión ayuda a entender la distribución sin modificarla de forma arbitraria.

## 4. Etiqueta legible para el tiempo de estudio

In [4]:
intervalos = {1: '<2', 2: '2-5', 3: '5-10', 4: '>10'}
preparados = {}
for nombre, df in datos.items():
    tabla = df.copy()
    tabla['studytime_hours'] = tabla['studytime'].map(intervalos)
    if tabla['studytime_hours'].isna().any():
        raise ValueError(f'Código de estudio inesperado en {nombre}')
    preparados[nombre] = tabla
    display(tabla[['studytime', 'studytime_hours', 'G3']].head())


,studytime,studytime_hours,G3
0,2,2-5,6
1,2,2-5,6
2,2,2-5,10
3,3,5-10,15
4,2,2-5,10


,studytime,studytime_hours,G3
0,2,2-5,11
1,2,2-5,11
2,2,2-5,12
3,3,5-10,14
4,2,2-5,13


La nueva columna expresa el mismo dato de manera legible. Para un modelo se elegiría una representación, evitando usar dos veces la misma información.

## 5. Exportación y comprobaciones

In [5]:
for nombre, tabla in preparados.items():
    ruta = resultados / f'tabla_minable_{nombre}.csv'
    tabla.to_csv(ruta, index=False, encoding='utf-8')
    comprobacion = pd.read_csv(ruta)
    assert len(comprobacion) == len(datos[nombre])
    assert comprobacion.isna().sum().sum() == 0
    assert 'G3' in comprobacion.columns
    print(f'{ruta.name}: {len(comprobacion)} filas, {len(comprobacion.columns)} columnas; sin nulos')


tabla_minable_matematicas.csv: 395 filas, 34 columnas; sin nulos
tabla_minable_portugues.csv: 649 filas, 34 columnas; sin nulos


Las tablas conservan las 33 variables originales y una etiqueta nueva. La selección final de predictores y las transformaciones para un modelo dependen de la pregunta y del momento en que se disponga de cada variable.